# Aruba population trends, 2015-2023


Source file: 1.11  Domiciliation by country of birth and sex (Aruba)\
Source: CBS Aruba and the Population Registry Office

---
## 1. Setup

## Imports and paths

This block of code is necessary to create a reproducible notebook environment.

In [1]:
# Importing necessary libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from config.project_paths import (
    ROOT,
    DATA_RAW,
    FIGURES
)

In [2]:
# Verify all paths to ensure stable environment

print("ROOT:", ROOT)
print("RAW DATA:", DATA_RAW)
print("FIGURES:", FIGURES)

ROOT: /home/ggirelli/Documents/DataAnalysis/projects/cbs_aruba
RAW DATA: /home/ggirelli/Documents/DataAnalysis/projects/cbs_aruba/data/raw
FIGURES: /home/ggirelli/Documents/DataAnalysis/projects/cbs_aruba/outputs/figures


In [3]:
DOMICIL_COUNTRY_SEX = DATA_RAW / "Table-1.11-Domiciliation-by-country-of-birth-and-sex.xlsx"

In [4]:
# # Stop early if the source file is missing

if not DOMICIL_COUNTRY_SEX.exists():
    raise FileNotFoundError

## 2. Load and inspect source table

In [5]:
raw_df = pd.read_excel(DOMICIL_COUNTRY_SEX, header=None)

In [6]:
print(raw_df.info())
print(raw_df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 19 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       10 non-null     str   
 1   1       9 non-null      object
 2   2       8 non-null      object
 3   3       9 non-null      object
 4   4       8 non-null      object
 5   5       9 non-null      object
 6   6       8 non-null      object
 7   7       9 non-null      object
 8   8       8 non-null      object
 9   9       9 non-null      object
 10  10      8 non-null      object
 11  11      9 non-null      object
 12  12      8 non-null      object
 13  13      9 non-null      object
 14  14      8 non-null      object
 15  15      9 non-null      object
 16  16      8 non-null      object
 17  17      9 non-null      object
 18  18      8 non-null      object
dtypes: object(18), str(1)
memory usage: 2.1+ KB
None
(12, 19)


In [7]:
years = raw_df.iloc[1].ffill()
sexes = raw_df.iloc[2]

columns = ["country"] + [
    f"{int(year)}_{sex.lower()}"
    for year, sex in zip(years.iloc[1:], sexes.iloc[1:])
]

df = raw_df.iloc[3:].copy()
df.columns = columns
df = df.dropna(how="all").reset_index(drop=True)

tidy_df = df.melt(
    id_vars="country",
    var_name="year_sex",
    value_name="value"
)

tidy_df[["year", "sex"]] = tidy_df["year_sex"].str.split("_", expand=True)
tidy_df = tidy_df.drop(columns="year_sex")

tidy_df["year"] = tidy_df["year"].astype(int)
tidy_df["sex"] = tidy_df["sex"].str.capitalize()
tidy_df["value"] = pd.to_numeric(tidy_df["value"], errors="coerce")

tidy_df

,country,value,year,sex
0,Total Domiciliation:,1780.0,2015,Male
1,Aruba/ Neth. Ant.,432.0,2015,Male
2,The Netherlands,370.0,2015,Male
3,Colombia,211.0,2015,Male
4,Venezuela,167.0,2015,Male
...,...,...,...,...
139,Colombia,274.0,2023,Female
140,Venezuela,363.0,2023,Female
141,Dominican Republic,71.0,2023,Female
142,Other,322.0,2023,Female
